# Backup Hard: executable workflow
This example creates synthetic data and runs inventory, policy locking, copy, and independent verification. It writes to a unique directory under `example-output/`.

In [ ]:
from pathlib import Path
import subprocess
import sys
import uuid

repo_root = Path.cwd()
if repo_root.name == 'examples':
    repo_root = repo_root.parent
run_root = repo_root / 'examples' / 'example-output' / uuid.uuid4().hex
source = run_root / 'source'
destination_root = run_root / 'destination'
reports_root = run_root / 'reports'
source.mkdir(parents=True)
destination_root.mkdir(parents=True)
(source / 'documents').mkdir()
(source / 'pictures').mkdir()
(source / 'documents' / 'notes.txt').write_text('portable example\n', encoding='utf-8')
(source / 'pictures' / 'sample.jpg').write_bytes(b'synthetic-jpeg-data')
backup_id = 'example_data_20260101'
run_root

Run each stage through the installed module entry points.

In [ ]:
def run_module(module, *arguments):
    command = [sys.executable, '-m', module, *map(str, arguments)]
    return subprocess.run(command, cwd=repo_root, check=True, text=True, capture_output=True)

inventory_result = run_module('backup_hard.inventory', '--backup-id', backup_id, '--source', source, '--reports-root', reports_root)
selection_result = run_module('backup_hard.select_formats', '--backup-id', backup_id, '--reports-root', reports_root, '--lock-all')
copy_result = run_module('backup_hard.copy_backup', '--backup-id', backup_id, '--source', source, '--destination-root', destination_root, '--reports-root', reports_root)
verification_result = run_module('backup_hard.verify_backup', '--backup-id', backup_id, '--source', source, '--destination-root', destination_root, '--reports-root', reports_root)
print(verification_result.stdout.strip())

In [ ]:
validation_report = reports_root / backup_id / 'step_4' / 'validation_report.md'
assert validation_report.is_file()
assert 'PASS' in validation_report.read_text(encoding='utf-8')
validation_report